In [49]:
import json 
import pandas as pd 
import gurobipy as gp

In [50]:
import pandas as pd



from src.build_recipe_matrix import load_recipe_data
from src.recipe_data import recipes

recipe_df, dishes, ingredients = load_recipe_data()
dishes = [dish for dish, _ in recipes]

import json

with open("./data/hp_data.json", "r") as file:
    hp_data = json.load(file)

In [52]:
MAX_HP = 50

hp = {
    dish: data["percent"] * MAX_HP + data["flat"]
    for dish, data in hp_data.items()
}

print(hp)

{'Sweet Madame': 910.0, 'Mondstadt Hash Brown': 615.0, 'Matsutake Meat Rolls': 910.0, 'Chicken-Mushroom Skewer': 804.0, 'Grilled Tiger Fish': 804.0, 'Universal Peace': 615.0, 'Squirrel Fish': 615.0, 'Crystal Shrimp': 1809.0, 'Barbeque Ribs': 910.0, 'Dry-Braised Salted Fish': 804.0, 'Duck Confit': 804.0, 'Fish With Cream Sauce': 1809.0, 'Grilled Fish in Mint Sauce': 804.0, 'Grilled Unagi Fillet': 910.0, 'Lakkaberry Madame': 910.0, 'Mint Jelly': 804.0, 'Pita Pocket': 615.0, 'Radish Veggie Soup': 1264.0, 'Smoked Fish Steak': 804.0, 'Soba Noodles': 1264.0, 'Tasses Ragout': 910.0, 'Tea-Smoked Squab': 910.0, 'Tonkotsu Ramen': 615.0, 'Tricolor Dango': 615.0, 'Tuna Sushi': 910.0, 'Veggie Pot Soup': 615.0, 'Poisson Seafood Soup': 2713.0, 'Golden Tempered Jade': 1810.0, 'Northern Apple Stew': 615.0, 'Crispy Potato Shrimp Platter': 1363.0, 'Tahchin': 1363.0, 'Flaming Red Bolognese': 1809.0, 'Bulle Soufle': 3734.0, 'Fine Tea, Full Moon': 615.0}


In [53]:

with open("../Genshin_HP_Food_Optimizer_LP_Model/data/inventory.json", "r") as file:
    Inventory = json.load(file)

In [54]:
import gurobipy as gp
model = gp.Model("Genshin Food production plan")

In [55]:

model = gp.Model("Genshin Food production plan")

dishes = [dish for dish, _ in recipes]



x = model.addVars(
    dishes,
    #vtype=gp.GRB.INTEGER,
    vtype=gp.GRB.CONTINUOUS,
    lb=0,
    name="x"
)

hp_value = {
    dish: data["percent"] * 100 + data["flat"]
    for dish, data in hp_data.items()
}

model.setObjective(
    gp.quicksum(
        hp_value[dish] * x[dish]
        for dish in dishes
    ), 
    gp.GRB.MAXIMIZE
)

# for ingredient in recipe_df.index:

#     model.addConstr(
#         gp.quicksum(
#             recipe_df.loc[ingredient, dish] * x[dish]
#             for dish in dishes
#         )
#         <= Inventory[ingredient],

#         name=f"{ingredient}_inventory"
#     )
constraints = {}

for ingredient in recipe_df.index:

    constraints[ingredient] = model.addConstr(
        gp.quicksum(
            recipe_df.loc[ingredient, dish] * x[dish]
            for dish in dishes
        )
        <= Inventory[ingredient],

        name=f"{ingredient}_inventory"
    )
model.optimize()
if model.status == gp.GRB.OPTIMAL:

    print("Maximum HP:", model.ObjVal)

    for dish in dishes:
        if x[dish].X > 0:
            print(
                dish,
                round(x[dish].X)
            )
    baseline_hp = model.ObjVal
    print("----- Shadow Prices--------------\n")
    for ingredient, constraint in constraints.items():
        print(
            ingredient, 
            round(constraint.Pi, 2)
        )

print("Baseline HP:", baseline_hp)
print("Variables:", model.NumVars)
print("Constraints:", model.NumConstrs)
# for ingredient in recipe_df.index:
#     for dish in recipe_df.columns:
#         coefficient = recipe_df.loc[ingredient, dish]
        

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model fingerprint: 0x898eb1f1
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]

Presolve removed 29 rows and 16 columns
Presolve time: 0.02s
Presolved: 9 rows, 18 columns, 28 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.8009733e+05   7.049488e+01   0.000000e+00      0s
       7    5.2888500e+05   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.03 seconds (0.00 work units)
Optimal objective  5.288850000e+05
Maximum HP: 528885.0
Mondstadt Hash Brown 10
Matsutake Meat Rolls 56
Chicken-Mushroom Skewer 110
Grilled Tiger Fish 18
Golden Tem

In [56]:
model.display()

Maximize
920.0 x[Sweet Madame] + 630.0 x[Mondstadt Hash Brown] + 920.0 x[Matsutake Meat Rolls]
+ 808.0 x[Chicken-Mushroom Skewer] + 808.0 x[Grilled Tiger Fish]
+ 1820.0 x[Golden Tempered Jade] + 630.0 x[Universal Peace] + 630.0 x[Squirrel Fish]
+ 1818.0 x[Crystal Shrimp] + 920.0 x[Barbeque Ribs] + 808.0 x[Dry-Braised Salted Fish]
+ 808.0 x[Duck Confit] + 1818.0 x[Fish With Cream Sauce]
+ 808.0 x[Grilled Fish in Mint Sauce] + 920.0 x[Grilled Unagi Fillet]
+ 920.0 x[Lakkaberry Madame] + 808.0 x[Mint Jelly] + 630.0 x[Pita Pocket]
+ 1268.0 x[Radish Veggie Soup] + 808.0 x[Smoked Fish Steak] + 1268.0 x[Soba Noodles]
+ 920.0 x[Tasses Ragout] + 920.0 x[Tea-Smoked Squab] + 630.0 x[Tonkotsu Ramen]
+ 630.0 x[Tricolor Dango] + 920.0 x[Tuna Sushi] + 630.0 x[Veggie Pot Soup]
+ 2726.0 x[Poisson Seafood Soup] + 3748.0 x[Bulle Soufle]
+ 1818.0 x[Flaming Red Bolognese] + 630.0 x[Northern Apple Stew]
+ 1376.0 x[Crispy Potato Shrimp Platter] + 1376.0 x[Tahchin]
+ 630.0 x[Fine Tea, Full Moon]
Subject To
  

/var/folders/pd/94n4w10j4xn8sstj1tsy8w2h0000gn/T/ipykernel_23955/259237463.py:1: DeprecationWarning: Model.display() is deprecated
  model.display()


In [57]:
for ingredient in recipe_df.index:

    used = sum(
        recipe_df.loc[ingredient, dish] * x[dish].X
        for dish in dishes
    )

    remaining = Inventory[ingredient] - used

    print(
        f"{ingredient:20s} "
        f"Inventory={Inventory[ingredient]:4.0f} "
        f"Used={used:4.0f} "
        f"Remaining={remaining:4.0f}"
    )

Apple                Inventory=   5 Used=   0 Remaining=   5
Bamboo Shoot         Inventory=  54 Used=   0 Remaining=  54
Berry                Inventory= 815 Used=   0 Remaining= 815
Bird Egg             Inventory=  17 Used=  10 Remaining=   7
Bulle                Inventory=  25 Used=  15 Remaining=  10
Butter               Inventory=   3 Used=   0 Remaining=   3
Cabbage              Inventory= 227 Used=   2 Remaining= 226
Carrot               Inventory=  75 Used=   2 Remaining=  74
Chenyu Adeptea       Inventory=  50 Used=   0 Remaining=  50
Fish                 Inventory=  18 Used=  18 Remaining=   0
Flour                Inventory=   5 Used=   5 Remaining=   0
Fowl                 Inventory= 176 Used= 176 Remaining=   0
Jam                  Inventory=  10 Used=  10 Remaining=   0
Lakkaberry           Inventory=  72 Used=   0 Remaining=  72
Lotus Head           Inventory=  97 Used=   0 Remaining=  97
Marcotte             Inventory=  59 Used=  16 Remaining=  42
Matsutake            Inv

### testing if 1 more ingridient is added to the inventory so 27 experiment to check how it improves

In [58]:
before = model.ObjVal

constraints["Fowl"].RHS = Inventory["Fowl"] + 1
model.optimize()

after = model.ObjVal

print("Before:", before)
print("After:", after)
print("Delta:", after - before)

constraints["Fowl"].RHS = Inventory["Fowl"]
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]

LP warm-start: use basis

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.2969300e+05   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.296930000e+05
Before: 528885.0
After: 529693.0
Delta: 808.0
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 

In [59]:
results = []

baseline_hp = model.ObjVal

for ingredient in Inventory:

    constraints[ingredient].RHS = Inventory[ingredient] + 1

    model.optimize()

    new_hp = model.ObjVal

    results.append({
        "ingredient": ingredient,
        "baseline_hp": baseline_hp,
        "new_hp": new_hp,
        "delta_hp": new_hp - baseline_hp
    })

    constraints[ingredient].RHS = Inventory[ingredient]

sensitivity_df = pd.DataFrame(results)

sensitivity_df = sensitivity_df.sort_values(
    "delta_hp",
    ascending=False
)

sensitivity_df

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]

LP warm-start: use basis

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.2888500e+05   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.288850000e+05
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 34 linear objective coefficients
Coefficient s

,ingredient,baseline_hp,new_hp,delta_hp
30,Sugar,528885.0,532609.0,3724.0
7,Flour,528885.0,530153.0,1268.0
15,Mint,528885.0,530153.0,1268.0
25,Salt,528885.0,529897.0,1012.0
6,Fish,528885.0,529693.0,808.0
8,Fowl,528885.0,529693.0,808.0
9,Jam,528885.0,529515.0,630.0
23,Rice,528885.0,529491.0,606.0
22,Raw Meat,528885.0,529345.0,460.0
28,Smetana,528885.0,529200.0,315.0


In [60]:
important_ingredients = [
    "Fish",
    "Fowl",
    "Rice",
    "Mushroom",
    "Potato",
    "Sugar",
    "Jam",
    "Flour",
    "Mint"
]
results = []

for ingredient in important_ingredients:

    for increase in range(1, 5):

        constraints[ingredient].RHS = Inventory[ingredient] + increase

        model.optimize()

        new_hp = model.ObjVal
        

        results.append({
            "ingredient": ingredient,
            "increase": increase,
            "baseline_hp": baseline_hp,
            "new_hp": new_hp,
            "delta_hp": new_hp - baseline_hp
        })
        print(
            increase,
            ingredient,
            baseline_hp,
            new_hp,
            
        )
    constraints[ingredient].RHS = Inventory[ingredient]
sensitivity_df = pd.DataFrame()
sensitivity_df = pd.DataFrame(results)

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]

LP warm-start: use basis

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.2969300e+05   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  5.296930000e+05
1 Fish 528885.0 529693.0
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model has 34 linear objective c

In [61]:
sensitivity_df

,ingredient,increase,baseline_hp,new_hp,delta_hp
0,Fish,1,528885.0,529693.000000,808.000000
1,Fish,2,528885.0,530501.000000,1616.000000
2,Fish,3,528885.0,531309.000000,2424.000000
3,Fish,4,528885.0,532117.000000,3232.000000
4,Fowl,1,528885.0,529693.000000,808.000000
5,Fowl,2,528885.0,530501.000000,1616.000000
6,Fowl,3,528885.0,531309.000000,2424.000000
7,Fowl,4,528885.0,532117.000000,3232.000000
8,Rice,1,528885.0,529491.000000,606.000000
9,Rice,2,528885.0,530097.000000,1212.000000


In [62]:
sensitivity_df["marginal_hp"] = (
    sensitivity_df
    .groupby("ingredient")["delta_hp"]
    .diff()
)